# 08 — Testes de Hipoteses

**Objetivos de aprendizagem:** formular hipóteses, executar testes e relatar evidência, efeito e limites

**Pré-requisitos:** Notebook 07

**Origem:** Slides, aula 10, páginas 201–221; aprofundamento: Welch, ANOVA, não paramétricos, multiplicidade e poder. Conteúdos adicionais estão identificados como **Aprofundamento**.

**Convenção:** `datasets/` reúne dados de entrada; `outputs/` reúne resultados. Dados inventados para demonstrações são sempre identificados como *sintéticos*.

### Materiais complementares

- [Guia técnico: Testes de Hipoteses](./reference_materials/08_testes_de_hipoteses.pdf)
- [Cheat sheet: Testes de Hipoteses](./cheat_sheets/cheat_sheet_testes_de_hipoteses.pdf)


## Preparação do ambiente

Execute a célula abaixo. Os notebooks são independentes e não dependem de variáveis definidas em outros arquivos.

> **Resultados reproduzíveis:** gráficos PNG e tabelas CSV selecionados são gravados automaticamente em `outputs/` ao executar as células na ordem. Os arquivos em `datasets/` são preservados. Esta disciplina usa Matplotlib, não o renderer `iframe` do Plotly.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Execute o JupyterLab com o diretório desta disciplina como raiz.
BASE = Path.cwd()
DATASETS = BASE / "datasets"
OUTPUTS = BASE / "outputs"
DATASETS.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

def ler_dataset(nome):
    caminho = DATASETS / nome
    if not caminho.is_file():
        raise FileNotFoundError(f"Dataset não encontrado: {caminho}. Abra o JupyterLab na raiz da disciplina.")
    return pd.read_csv(caminho)

RNG = np.random.default_rng(42)

## 1. H0, H1, significância e p-valor

O p-valor é calculado sob H0 e não é P(H0|dados). Um teste não mede magnitude de efeito sozinho. Informe hipótese, teste, p-valor, intervalo e tamanho de efeito quando possível.

In [2]:
from scipy import stats
A=RNG.normal(50,10,60);B=RNG.normal(54,11,65) # grupos sintéticos
t=stats.ttest_ind(A,B,equal_var=False)
print('Welch t:',t.statistic,'p:',t.pvalue,'diferença B-A:',B.mean()-A.mean())

Welch t: -1.1691332085637225 p: 0.244611058062067 diferença B-A: 1.7233285861057013


## 2. Independentes vs pareados

Welch compara médias independentes sem assumir variâncias iguais. Para medidas repetidas nos mesmos indivíduos, a unidade relevante é a diferença pareada.

In [3]:
antes=RNG.normal(100,12,30);depois=antes-3+RNG.normal(0,4,30)
tpar=stats.ttest_rel(antes,depois)
print('teste t pareado:',tpar.statistic,tpar.pvalue)

teste t pareado: 3.7852012228387353 0.0007145033182846099


## 3. ANOVA e alternativas (aprofundamento)

ANOVA de uma via avalia igualdade das médias de três ou mais grupos sob pressupostos apropriados; um resultado significativo não informa quais pares diferem. Kruskal–Wallis compara distribuições/ranks e não é um teste universal de medianas.

In [4]:
g1=RNG.normal(10,2,35);g2=RNG.normal(11,2,35);g3=RNG.normal(13,2,35)
print('ANOVA:',stats.f_oneway(g1,g2,g3))
print('Kruskal:',stats.kruskal(g1,g2,g3))

ANOVA: F_onewayResult(statistic=np.float64(22.129378955757602), pvalue=np.float64(1.0408612341721289e-08))
Kruskal: KruskalResult(statistic=np.float64(34.22065460146325), pvalue=np.float64(3.7074838657024565e-08))


## 4. Qui-quadrado em tabelas de contingência

O teste avalia independência entre variáveis categóricas. Verifique frequências esperadas, desenho da amostra e contagens independentes; para 2×2 com poucos dados, considere teste exato de Fisher.

In [5]:
obs=np.array([[30,20],[15,35]])
chi,p,gl,expected=stats.chi2_contingency(obs,correction=False)
print('qui²',chi,'p',p,'gl',gl,'esperadas',expected)
assert (expected>0).all()
print('Fisher exato (2x2):',stats.fisher_exact(obs))

qui² 9.09090909090909 p 0.0025688315270227164 gl 1 esperadas [[22.5 27.5]
 [22.5 27.5]]
Fisher exato (2x2): SignificanceResult(statistic=np.float64(3.5), pvalue=np.float64(0.004635020180798911))


In [6]:
# Exportação reproduzível para outputs/ (dados e resultados didáticos)
pd.DataFrame(obs,index=['linha_1','linha_2'],columns=['coluna_1','coluna_2']).to_csv(OUTPUTS / '08_tabela_contingencia_observada.csv',encoding='utf-8-sig')
pd.DataFrame(expected,index=['linha_1','linha_2'],columns=['coluna_1','coluna_2']).to_csv(OUTPUTS / '08_frequencias_esperadas.csv',encoding='utf-8-sig')
print('Resultados exportados em:', OUTPUTS)

Resultados exportados em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/03_statistics/outputs


## 5. Tamanho de efeito e erros de decisão

Para grupos independentes, d de Cohen divide diferença de médias pelo desvio padrão agrupado (se interpretável). Erro tipo I rejeita H0 verdadeira; tipo II não a rejeita quando falsa. Poder = 1−β.

In [7]:
sp=np.sqrt(((len(A)-1)*A.var(ddof=1)+(len(B)-1)*B.var(ddof=1))/(len(A)+len(B)-2))
d=(B.mean()-A.mean())/sp
print('d de Cohen aproximado:',d)

d de Cohen aproximado: 0.2085261724279157


## 6. Múltiplas comparações (aprofundamento)

Se você testa muitos desfechos, a taxa de falsos positivos aumenta. Bonferroni controla o erro familiar; Benjamini–Hochberg controla FDR sob condições específicas.

In [8]:
from statsmodels.stats.multitest import multipletests
ps=np.array([.001,.012,.03,.07,.20])
for metodo in ['bonferroni','fdr_bh']:
 rejeita,ajustados,_,_=multipletests(ps,alpha=.05,method=metodo)
 print(metodo,'rejeições',rejeita,'p ajustados',np.round(ajustados,3))

bonferroni rejeições [ True False False False False] p ajustados [0.005 0.06  0.15  0.35  1.   ]
fdr_bh rejeições [ True  True  True False False] p ajustados [0.005 0.03  0.05  0.088 0.2  ]


In [9]:
# Exportação reproduzível para outputs/ (dados e resultados didáticos)
pd.DataFrame({'p_original':ps,'p_bonferroni':multipletests(ps,method='bonferroni')[1],'p_fdr_bh':multipletests(ps,method='fdr_bh')[1]}).to_csv(OUTPUTS / '08_correcao_multiplos_testes.csv',index=False,encoding='utf-8-sig')
print('Resultados exportados em:', OUTPUTS)

Resultados exportados em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/03_statistics/outputs


### Exercício resolvido — Escolha do teste para dois grupos

**Solução comentada:**

In [10]:
exemplo=pd.DataFrame({'situacao':['duas turmas independentes','antes e depois nos mesmos alunos','duas categorias em grupos'],
                   'avaliacao':['Welch t (se aplicável)','t pareado (se aplicável)','qui-quadrado/Fisher (conforme condições)']})
display(exemplo)

,situacao,avaliacao
0,duas turmas independentes,Welch t (se aplicável)
1,antes e depois nos mesmos alunos,t pareado (se aplicável)
2,duas categorias em grupos,qui-quadrado/Fisher (conforme condições)


In [11]:
# Exportação reproduzível para outputs/ (dados e resultados didáticos)
pd.DataFrame({'teste':['Welch independente','t pareado','ANOVA','Kruskal','qui_quadrado'], 'estatistica':[t.statistic,tpar.statistic,stats.f_oneway(g1,g2,g3).statistic,stats.kruskal(g1,g2,g3).statistic,chi], 'p_valor':[t.pvalue,tpar.pvalue,stats.f_oneway(g1,g2,g3).pvalue,stats.kruskal(g1,g2,g3).pvalue,p]}).to_csv(OUTPUTS / '08_resumo_testes.csv',index=False,encoding='utf-8-sig')
print('Resultados exportados em:', OUTPUTS)

Resultados exportados em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/03_statistics/outputs


**Interpretação:** A árvore de decisão começa pela unidade de análise e natureza da variável; pressupostos, tamanho de amostra e objetivo refinam a escolha.

## Erros comuns e limites

Falhar em rejeitar H0 não prova equivalência; p pequeno não implica efeito importante. Não selecione testes apenas pela aparência de normalidade em histogramas.

## Consulta Rápida

`stats.ttest_ind(x,y,equal_var=False)` · `stats.ttest_rel(x,y)` · `stats.f_oneway(a,b,c)` · `stats.chi2_contingency(tab)` · `stats.fisher_exact(tab)` · `multipletests(pvals,method="fdr_bh")`.

## Autoavaliação

Antes de seguir, explique com suas palavras: **qual é a pergunta estatística, qual método responde a ela, quais pressupostos devem ser verificados e o que não podemos concluir?**

## Aprofundamento v2 — Intervalos de diferenças, efeito e múltiplos testes

Uma decisão por `p < 0,05` não informa a magnitude ou relevância prática do efeito. Para grupos independentes com variâncias possivelmente distintas, o teste de Welch e o intervalo de confiança da diferença devem ser interpretados em conjunto. Correções de multiplicidade limitam erros ao testar múltiplas hipóteses relacionadas, mas não corrigem seleção enviesada de resultados.

In [12]:
from scipy import stats
from statsmodels.stats.multitest import multipletests
rng_welch=np.random.default_rng(2026)
a_w=rng_welch.normal(10,2,45)
b_w=rng_welch.normal(11,3,50)
dif=b_w.mean()-a_w.mean()
se=np.sqrt(a_w.var(ddof=1)/len(a_w)+b_w.var(ddof=1)/len(b_w))
num=(a_w.var(ddof=1)/len(a_w)+b_w.var(ddof=1)/len(b_w))**2
den=(a_w.var(ddof=1)/len(a_w))**2/(len(a_w)-1)+(b_w.var(ddof=1)/len(b_w))**2/(len(b_w)-1)
gl=num/den
tcrit=stats.t.ppf(.975,gl)
print('Diferença de médias B-A:',round(dif,3),'IC95% Welch:',np.round([dif-tcrit*se,dif+tcrit*se],3))
ps_teste=np.array([.004,.018,.04,.08])
print('FDR-BH:',np.round(multipletests(ps_teste,method='fdr_bh')[1],3))
assert se>0

Diferença de médias B-A: 0.963 IC95% Welch: [-0.141  2.067]
FDR-BH: [0.016 0.036 0.053 0.08 ]


### Exercício resolvido

**Pergunta:** um estudo encontrou `p = 0,003`, mas a diferença observada é de 0,01 unidade, abaixo da diferença mínima considerada relevante. A conclusão é “grande benefício”?

**Solução:** não. Há evidência contra a hipótese nula sob os pressupostos do teste, mas a importância prática requer tamanho de efeito, IC, custos, contexto e qualidade do desenho.